# Precision-Recall Curves

Evaluate model performance with PR curves and threshold analysis.

In [ ]:
import pickle
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score
from pathlib import Path

# Load model
model_path = Path('../backend/ml/artifacts/model.pkl')
if model_path.exists():
    with open(model_path, 'rb') as f:
        model = pickle.load(f)
    print('Model loaded successfully')
else:
    print('Model not found. Run `make train` first.')

In [ ]:
# Load and prepare test data
import sys
sys.path.insert(0, '../backend')
from ml.train import load_data, create_splits

df = load_data('../backend/data/output')
train, val, test = create_splits(df)

feature_cols = ['total_orders', 'total_refunds', 'total_amount', 'avg_amount',
                'max_amount', 'refund_rate', 'refund_ratio', 'high_amount']

X_test = test[feature_cols].values
y_test = test['label'].values

print(f'Test samples: {len(y_test)}')
print(f'Positive rate: {y_test.mean():.2%}')

## PR Curve

In [ ]:
# Compute PR curve
y_proba = model.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, 'b-', linewidth=2, label=f'XGBoost (PR-AUC = {pr_auc:.4f})')
plt.fill_between(recall, precision, alpha=0.1, color='blue')
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curve - EncryptionGuard v5', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.xlim([0, 1])
plt.ylim([0, 1.05])
plt.tight_layout()
plt.savefig('03_pr_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'PR-AUC: {pr_auc:.4f}')

## Threshold Analysis

In [ ]:
# Find optimal threshold
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-8)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

plt.figure(figsize=(8, 6))
plt.plot(thresholds, precision[:-1], 'b-', label='Precision')
plt.plot(thresholds, recall[:-1], 'r-', label='Recall')
plt.plot(thresholds, f1_scores, 'g--', label='F1 Score')
plt.axvline(x=optimal_threshold, color='k', linestyle=':', label=f'Optimal threshold = {optimal_threshold:.3f}')
plt.xlabel('Threshold', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Precision, Recall, and F1 vs Threshold', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('03_threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Optimal threshold: {optimal_threshold:.3f}')
print(f'F1 at optimal: {f1_scores[optimal_idx]:.4f}')